# 추천시스템

## Part 1. 추천 시스템 기초
### 1. 추천 시스템이란?
- 사용자의 행동 기록과 아이템 정보를 바탕으로, 사용자가 좋아할 만한 아이템을 예측해 골라 주는 시스템
- 필요한 이유
    - 고객 : 찾는 시간과 노력 감소, 몰랐던 취향의 아이템 발견
    - 기업 : 이용 시간, 구매 전환, 재방문 증가, 비인기 아이템(롱테일) 노출
- 검색은 이미 아는 것을 찾게 돕고, 추천은 아직 모르는 것을 발견하게 도움

#### 1-1. 사용되는 데이터의 종류와 특징
- 명시적 데이터(Explicit Data) : 사용자가 선호를 직접 표현한 데이터 -> ex. 별점, 좋아요, 리뷰
    - 장점 : 선호와 불호가 분명함
    - 단점 : 평가를 남기는 사람 자체가 적음, 취향이 변함, 사람마다 점수 기준이 다름
- 암시적 데이터(Implicit Data) : 행동에서 선호를 간접적으로 추측하는 데이터 -> ex. 클릭, 구매, 시청 시간
    - 장점 : 따로 입력하지 않아도 대량으로 쌓임
    - 단점 : 행동이 곧 선호라고 단정할 수 없음 (행동이 있어도 좋아한다는 보장 X, 없어도 싫어한다는 보장 X)
- 실제 서비스에서는 두 가지를 섞어 쓰는 경우가 대부분
- 추천 시스템의 종류
    - 내용 기반 필터링 : 아이템 유사도를 근거로
    - 협업 필터링 : 다수 사용자의 행동 패턴을 근거로
        - 사용자 기반 : 나와 비슷한 사용자가 선호한 아이템 추천
        - 아이템 기반 : 내가 선호한 아이템과 비슷한 아이템 추천
    - 하이브리드 필터링 : 두 가지 이상을 조합해 각각의 단점 보완
        - Weighted, Mixed, Switch, Feature Combination, Meta-Level
        - ex. 새 영화는 내용 기반으로, 평점이 쌓이면 협업 필터링으로 -> Switch


### 2. 내용 기반 필터링 (Content-Based Filtering)
- 아이템을 숫자로 표현 -> 사용자 프로필 생성 -> 유사도 계산 -> 점수가 높은 아이템 추천

#### 2-1. 사용자 프로필
- 사용자가 어떤 특성을 좋아하는지 정리한 정보
- 직접 지정형 : 사용자가 취향을 직접 알려 줌 / 간접 지정형 : 과거 행동에서 취향을 추론
- 프로필 획득 방식
    1. 사용자가 본 아이템들의 특성 가중치(0~1) 평균
    2. 설문, 키워드 입력, 평가 등으로 기호 파악 (명시적, 암시적 피드백)
    3. 유저의 기존 평가를 정답 라벨로 써서 분류 문제로 해결 -> 결정 트리, 랜덤 포레스트, KNN 등
- 좋은 사용자 프로필을 구성할수록 추천 정확도와 만족도가 높아짐

#### 2-2. 유사도 (Similarity)
- 두 대상이 얼마나 비슷한지 나타내는 숫자
- 사용자와 아이템을 같은 feature로 표현해야 비교 가능

    <img src="https://developers.google.com/machine-learning/recommendation/images/Matrix1.svg" width="450" style="background-color:white; padding:10px;">

- 과정
    1. 아이템을 숫자로 변환 - 장르, 감독 같은 feature를 원-핫 인코딩
    2. 사용자를 숫자로 변환 - 아이템과 같은 n개의 feature로 표현
    3. 유사도 계산 - 내적이 클수록 유사
- 유사도 종류 : 자카드 유사도(집합), 코사인 유사도(각도), 유클리디안 거리(거리), 피어슨 상관계수(상관관계)
- 코사인 유사도 : 두 벡터 사이의 각도로 측정, 1에 가까울수록 비슷

$$ \cos(A, B) = \frac{A \cdot B}{\|A\| \|B\|} $$

- 길이는 무시하고 방향만 봄 -> ex. [1, 1]과 [5, 5]는 코사인 유사도 1
- 내적 vs. 코사인
    - 내적 : 길이까지 반영 -> feature가 많은 아이템이 유리해짐
    - 코사인 : 길이 무시 -> 쏠림은 줄지만 아이템 정보가 풍부한 점은 반영 X

#### 2-3. TF-IDF (Term Frequency - Inverse Document Frequency)
- 특정 문서에는 자주 나오고 다른 문서에는 드문 단어에 높은 값을 주는 방법
- 단순 단어 빈도는 "그리고", "있다" 같은 단어가 중요하게 취급되는 문제가 있어서 등장
- TF : 특정 문서에서 단어가 등장한 횟수
- IDF : 그 단어가 등장한 문서 수가 많을수록 작아짐

$$ IDF(d, t) = \log\left(\frac{n}{1 + DF(t)}\right), \quad TF\text{-}IDF = TF(d, t) \times IDF(d, t) $$

- 추천에서는 줄거리를 TF-IDF 벡터로 바꾸고 코사인 유사도로 비교해서 추천

#### 2-4. 내용 기반 필터링의 장단점
- 장점
    - 아이템 콜드 스타트 해결 -> 속성만 있으면 신규 아이템도 추천 가능
    - 다른 사용자의 데이터 필요 X
    - 추천 이유 설명 용이
- 단점
    - feature로 담을 수 있는 정보에 한계
    - 비슷한 것만 계속 추천 (과도한 특화, Over-specialization)
    - 사용자 콜드 스타트는 해결 어려움
- 콜드 스타트(Cold start) : 충분한 정보가 없어 적절한 추천을 못하는 문제


### 3. 협업 필터링 (Collaborative Filtering)
- 가정 : 과거에 비슷한 선호도를 가진 사용자들은 앞으로도 비슷한 선호도를 가질 가능성이 높다

#### 3-1. 메모리 기반 협업 필터링 (Memory-Based Collaborative Filtering)
- 저장된 사용자-아이템 평점을 그대로 꺼내서 새 아이템의 평점을 예측하는 방법
- 별도의 학습 없이 "이웃"을 찾아 씀 -> 이웃 기반 협업 필터링이라고도 함
- 사용자 기반(User-based)
    - 나와 유사도가 가장 높은 TOP-N 사용자를 이웃으로 선정
    - 이웃들의 평점을 유사도로 가중평균해서 예측
    - 평점 데이터에서는 유사도로 피어슨 상관계수를 많이 사용

$$ \hat{r}_{ui} = \bar{r}_u + \frac{\sum_v sim(u, v)(r_{vi} - \bar{r}_v)}{\sum_v |sim(u, v)|} $$

- 아이템 기반(Item-based)
    - 사용자들이 함께 좋아하고 싫어한 패턴으로 아이템 유사도 판단
    - 아이템끼리의 관계는 덜 변함 -> 미리 계산해 두고 재사용하기 좋음

| 기준 | 사용자 기반 | 아이템 기반 |
|:---:|:---:|:---:|
| 정확성 | 아이템 수가 사용자 수보다 많을 때 유리 | 사용자 수가 아이템 수보다 많을 때 유리 |
| 계산 속도 | 추천할 때마다 이웃을 새로 찾음 | 미리 계산해 두고 재사용 -> 빠름 |
| 다양성 | 의외성, 참신성이 강점 | 다양성 상대적으로 부족 |

- 새로운 추천이 필요한 플랫폼 -> User-based / 정확하고 안정적인 추천이 중요한 대형 플랫폼 -> Item-based

#### 3-2. 메모리 기반 협업 필터링의 한계
- 희소 행렬 문제
    - 실제 사용자-아이템 행렬은 대부분 비어 있음 -> ex. 넷플릭스 프라이즈 데이터 약 99%가 빈칸
    - 함께 평가한 아이템이 적어서 유사도를 믿기 어려움
    - 해결 -> 모델 기반 협업 필터링
- 콜드 스타트 문제
    - 신규 사용자 : 이웃을 찾을 수 없음
    - 신규 아이템 : 아무도 평가하지 않아 추천 후보에 못 오름 (first-rater 문제)
    - 해결 -> 가입 시 선호 장르 선택, 하이브리드 방식, 같은 카테고리 임베딩 평균, 인구통계학적 정보 활용
- 확장성 문제(Scalability)
    - 사용자와 아이템이 늘어날수록 계산량 증가
    - 해결 -> KNN 협업 필터링 : 유사도가 높은 k명의 이웃만 골라서 예측
    - k가 너무 작으면 소수 의견에 흔들리고, 너무 크면 예측이 흐려짐

#### 3-3. 모델 기반 협업 필터링 (Model-Based Collaborative Filtering)
- 관측된 평점으로 예측 모델을 학습하고, 관측되지 않은 평점을 예측하는 방법
- 학습 단계(미리 수행) -> 예측 단계(추천 시점에 빠르게 수행)
- 평점 이면의 잠재된 패턴을 학습 -> 직접 겹치는 평가가 없어도 빈칸 추정 가능
- 잠재 요인(Latent Factor) : 평점에 직접 드러나지는 않지만 사용자와 아이템의 관계를 결정짓는 숨은 특성
- Latent Factor Model : 사용자와 아이템을 같은 잠재 공간의 벡터로 나타내고, 두 벡터의 관계로 평점을 설명
    - 축의 개수(k)만 사람이 정하고, 축의 의미는 학습으로 정해짐 -> 해석이 어려움

    <img src="https://developers.google.com/machine-learning/recommendation/images/2D.svg" width="400" style="background-color:white; padding:10px;">

- 행렬 분해(Matrix Factorization) : 큰 평점 행렬을 작은 행렬 두 개의 곱으로 쪼개는 방법

    <img src="https://developers.google.com/machine-learning/recommendation/images/Matrixfactor.svg" width="450" style="background-color:white; padding:10px;">

$$ \hat{R} = P^T Q, \quad \hat{r}_{ui} = p_u^T q_i $$

- 학습 과정
    1. 잠재 요인 개수 k 정하기
    2. 벡터를 임의의 값으로 시작
    3. 평점이 있는 칸만 예측하고 실제 평점과의 오차 계산
    4. 오차가 줄어들도록 벡터 수정 (SGD, ALS)
    5. 3~4 반복 후 빈칸 예측
- 손실 함수 = 오차 + 규제(λ) -> 규제로 과적합 방지
    - SGD : 평점을 하나씩 보면서 조금씩 수정
    - ALS : 사용자 벡터와 아이템 벡터를 번갈아 고정 -> 병렬 처리에 유리
- 편향(Bias) : 사람마다 점수 기준이 다르고 작품마다 평이 다른 걸 반영

$$ \hat{r}_{ui} = \mu + b_i + b_u + p_u^T q_i $$

- 행렬 분해 방식 : SVD, 학습 기반 MF(SGD, ALS), NMF, PMF
- 메모리 기반 vs. 모델 기반
    - 메모리 기반 : 비슷한 사용자와 아이템을 직접 찾음, 추천 이유 설명 쉬움
    - 모델 기반 : 학습한 벡터를 내적, 빠르고 일반화에 유리
    - 둘 다 콜드 스타트는 해결 어려움
- 한계 : 결국 내적이라는 선형 계산으로 예측


## Part 2. 딥러닝 기반 추천 시스템
### 1. VAE 기반 협업 필터링 (Variational Autoencoder CF)
- 사용자의 상호작용 기록을 확률분포로 압축한 뒤, 다시 복원하면서 아직 보지 않은 아이템의 선호를 예측하는 모델
- 인코더 -> 평균 μ와 표준편차 σ 출력 / 샘플링 -> 잠재 벡터 z / 디코더 -> 아이템별 선호 확률 출력
- 대표 모델 : Mult-VAE

#### 1-1. MF의 한계와 딥러닝의 등장
- 선형성 : 내적은 요인별로 곱해서 더하기만 함 -> 요인끼리 얽히는 효과 표현 어려움
    - 해법 : 비선형 활성화 함수를 가진 신경망
- 불확실성 무시 : 사용자마다 벡터를 하나만 학습(점 추정) -> 기록이 적은 사용자도 똑같이 확신
    - 해법 : 평균과 분산을 함께 출력

#### 1-2. 왜 AutoEncoder(AE)가 아닌 VAE(Variational AE)인가?
- AE : 입력을 작은 벡터로 압축했다가 복원하는 신경망
    - AutoRec : AE를 협업 필터링에 적용한 초기 모델 -> 비선형성이 추천 성능을 높일 수 있음을 보여줌
    - but 사용자를 점 하나로 표현한다는 점은 MF와 같음
- VAE : 잠재 벡터를 점 하나가 아닌 분포로 표현
    - 평균 μ : 사용자 취향의 중심 위치
    - 표준편차 σ : 그 위치에 대한 불확실성

    <img src="https://lilianweng.github.io/posts/2018-08-12-vae/vae-gaussian.png" width="450" style="background-color:white; padding:10px;">

- VAE가 더 안정적인 이유
    1. 점 하나가 아니라 그 주변에서 샘플링해서 학습
    2. KL 항이 분포를 표준 정규분포 쪽으로 당김 -> 기록 몇 개만으로 극단적인 위치로 가기 어려움

#### 1-3. VAE 기반 CF (Mult-VAE)의 작동 구조
- 입력 : 아이템 수만큼 긴 0/1 벡터 (평점 4점 이상만 1)
- 인코더 : 평균 μ와 표준편차 σ 출력
- 잠재 벡터 z : μ, σ로 정해진 정규분포에서 샘플링
- 디코더 : 모든 아이템의 점수를 계산하고 softmax로 확률 변환

#### 1-4. VAE의 비밀 병기: 목적 함수와 KL Divergence
- ELBO(Evidence Lower BOund)를 최대화

$$ L(x; \theta, \phi) = E_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \| p(z)) $$

- 복원 항 : 원래 입력을 최대한 비슷하게 복원
- KL Divergence 항 : 잠재 분포가 표준 정규분포에서 크게 벗어나지 않도록 -> 과적합 방지, 잠재 공간을 부드럽게
- KL Divergence : 두 확률분포가 얼마나 다른지 나타내는 값
    - 중심에서 멀어지거나, 지나치게 확신하거나, 지나치게 퍼지면 커짐
- AE vs. VAE 잠재 공간
    - AE : 벡터들이 제약 없이 흩어짐 -> 기록이 적은 사용자가 엉뚱한 곳에 놓이기 쉬움
    - VAE : 분포들이 중앙 근처에 연속적으로 모임 -> 비슷한 사용자들 근처에 안정적으로 놓임
- 장점 : 불확실성 반영, 희소한 데이터에서도 안정적, 비선형성 학습


### 2. GNN 기반 사용자 선호 예측
- CF, MF, VAE는 상호작용을 행렬의 칸 하나하나로 다룸 -> 여러 상호작용이 이어진 관계는 직접 활용 X

#### 2-1. GNN (Graph Neural Network, 그래프 신경망)
- 사용자-아이템 상호작용은 본질적으로 그래프 -> 사용자와 아이템은 점, 행동은 선
- 나 -> 영화 X -> (X를 본) 다른 사용자 -> 영화 Y 같은 경로로 추천 가능
- 그래프 : 노드(Node)와 엣지(Edge)로 구성한 자료 구조
    - 노드 : 개체 (사용자, 아이템 등)
    - 엣지 : 노드 사이의 관계, 방향과 가중치를 가질 수 있음

    <img src="https://distill.pub/2021/gnn-intro/directed_undirected.e4b1689d.png" width="400" style="background-color:white; padding:10px;">

- 그래프를 쓰는 이유
    - 관계, 상호작용 같은 추상적인 개념을 다루기에 적합
    - Non-Euclidean space도 표현 가능 -> 노드마다 이웃 수가 제각각이라 CNN, RNN 대신 GNN 필요

#### 2-2. GNN의 구조
1. 그래프 구조 생성 - 노드와 엣지 정의, 초기 피처(임베딩) 부여
2. 메시지 패싱 - 메시지 생성 -> 집계 -> 갱신
3. 최종 출력
- 핵심 아이디어 : 이웃 노드의 정보를 나에게 전달/요약 (Message Passing)

#### 2-3. GNN의 프로세스
- 그래프 구조 생성
    - 인접 행렬(Adjacency matrix) : 노드 간 연결 관계 표현
    - 피처 행렬(Feature matrix) : 노드 고유의 feature 표현
- 메시지 패싱 (레이어 하나 = 메시지 생성 - 집계 - 갱신)
    - 메시지 생성 : 이웃 노드가 자신의 임베딩으로 메시지 생성
    - 집계 : 받은 메시지를 하나의 벡터로 합침 (sum, mean, max, attention 등)
    - 갱신 : 집계 결과와 자신의 이전 임베딩을 결합해 새 임베딩 계산

    <img src="https://distill.pub/2021/gnn-intro/arch_gcn.40871750.png" width="450" style="background-color:white; padding:10px;">

- 이 과정을 K번 반복 -> K단계 떨어진 이웃의 정보까지 반영
- 모든 노드가 한 번씩 타깃 노드가 됨

#### 2-4. LightGCN (Light Graph Convolutional Networks)
- GNN 아키텍처의 차이 : 메시지를 어떻게 만들고, 어떻게 합치는가
    - GCN : 이웃 임베딩을 차수로 정규화해 평균, 가장 기본 구조
    - GRN : RNN + GNN, 시간에 따라 변하는 그래프 포착
    - GAT : attention으로 이웃마다 중요도를 학습해 가중 평균
    - GraphSAGE : 이웃을 일부만 샘플링, 자기 정보와 concat -> 대규모 그래프에 적합

    <img src="https://distill.pub/2021/gnn-intro/attention.3c55769d.png" width="350" style="background-color:white; padding:10px;">

- 추천에서는 NGCF(2019) -> 이를 단순화한 LightGCN(2020) 등장
- LightGCN 탄생 배경
    - 추천 그래프의 노드에는 ID 임베딩만 있음
    - NGCF의 가중치 행렬 W와 비선형 활성화 함수가 성능에 도움이 안 되고 오히려 학습을 방해
- 제거한 것
    1. 특징 변환(Feature Transformation) -> 가중치 행렬 W 삭제
    2. 비선형 활성화 함수 -> 정규화된 합 혹은 평균만 사용
- "정규화된 이웃 임베딩의 합 + 레이어 결합"만으로 구성 -> NGCF보다 평균 약 16% 높은 성능

#### 2-5. GNN 활용 사례
- Pinterest : 핀과 보드를 노드로 한 약 30억 노드, 180억 엣지 그래프 -> GraphSAGE처럼 이웃 일부만 샘플링해서 학습
- 그 외 : 소셜 네트워크, 화학 및 생명과학(분자 구조), 금융 및 보안(이상 거래 탐지), 교통, 프로그램 분석

#### 2-6. GNN의 장단점
- 장점
    - 고차원 협업 시그널 포착 -> 이웃과 연관 아이템 정보까지 반영
    - 정형화되지 않은 데이터 학습 가능
    - Cold Start 완화
- 단점
    - 과평탄화(Over-smoothing) : 레이어가 많아질수록 모든 노드의 임베딩이 비슷해짐
    - 과압축(Oversquashing) : 늘어나는 이웃 정보를 고정 크기 벡터에 담으면서 먼 노드 정보 손실
    - 계산 복잡도 증가
    - 순서 반영의 어려움


### 3. Transformer 기반 추천 시스템
#### 3-1. 기존 추천 시스템의 순차적 한계
- CF : 순서 무시, '방금'과 '1년 전' 구매의 중요도 구분 X, 일반적인 선호만 예측
- GNN : 정보가 여러 단계를 거치며 희석됨, 레이어 수만큼만 볼 수 있어서 장기 의존성 문제
- CF는 '누가 무엇을', GNN은 '무엇이 무엇과' -> 둘 다 '어떤 순서로, 지금 어떤 의도로'는 못 봄

#### 3-2. Transformer: 사용자 행동을 순서(Sequence)로 접근
- 다음 행동은 장기적인 평균 취향보다 최근 행동의 순서와 조합(문맥)에 더 영향을 받음
- 문맥에 따라 같은 아이템도 의미가 달라짐
    - ex. 수영복 -> 비치타월 -> 선크림 : 휴가 준비물 / 스킨 -> 로션 -> 선크림 : 스킨케어 단계
- 거리와 관계없이 핵심 의도 포착 (Long-Range Dependency)
    - ex. 프린터 -> 양말 -> 라면 -> 폰케이스 -> 생수 -> 샴푸 -> 토너 추천

#### 3-3. Transformer의 핵심: Self-Attention
- 시퀀스 안의 모든 아이템 쌍의 관련성을 거리와 상관없이 동시에 계산 -> 장기 의존성 문제 해결
- Query : 관련성을 계산하는 기준 / Key : Query와 내적되어 점수 결정 / Value : 가중합되는 실제 정보

$$ Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d}}\right)V $$

1. Attention Score 계산 - Query와 각 Key 내적
2. Softmax - 합이 1인 가중치로 변환
3. 가중합 - 각 Value에 가중치를 곱해 더함

#### 3-4. Transformer 기반 주요 추천 시스템 모델
- SASRec
    - 단방향 어텐션으로 과거 정보만 활용해서 다음 아이템 예측 (GPT와 유사)
    - 특징 : 데이터가 희소하면 최근 아이템에, 풍부하면 긴 과거 맥락에 집중 / 병렬 처리로 빠른 학습
    - 문제점 : 연산량이 시퀀스 길이의 제곱에 비례 / 단방향이라 앞부분 아이템은 활용할 정보가 적음
- BERT4Rec
    - 양방향 어텐션으로 과거와 미래 정보를 모두 활용해서 가려진 아이템 예측 (BERT와 유사)
    - [MASK]로 가리는 이유 : 양방향 학습 시 target 아이템 정보를 직접 보는 정보 누출 방지
    - 특징 : 양방향 문맥 학습 / 가리는 위치를 바꿔 여러 학습 샘플 생성 가능

| 구분 | SASRec | BERT4Rec |
|:---:|:---:|:---:|
| 어텐션 방향 | 단방향 (과거만) | 양방향 (과거 + 미래) |
| 학습 목표 | 다음 아이템 예측 | 가려진 아이템 예측 |
| 학습 효율 | 빠름 | 상대적으로 느림 |

- BERT4Rec이 항상 더 좋은 건 X -> 같은 손실 함수로 맞추면 SASRec이 비슷하거나 더 높은 성능
    - 상호작용이 많고 '다음 행동' 예측이 목적 -> SASRec
    - 행동 순서가 느슨한 도메인(장바구니 등) -> BERT4Rec


### 4. 제언: 오늘날의 추천 시스템
#### 4-1. LLM의 시대
- 추천을 랭킹 문제가 아닌 언어 문제로 풂 -> 점수 대신 문장으로 출력
- ID가 아닌 의미(Semantic) 기반, 멀티모달 데이터 활용
- 추천 이유를 함께 제시하는 서술형 추천 가능
- 대화형 추천 : 자연어로 입력한 조건 반영
- Zero-shot 추천 : 상호작용 데이터 없이 텍스트 설명만으로 추천 -> Cold-Start 해결책
- P5 : 여러 추천 과제를 "텍스트 프롬프트 -> 텍스트 답변"으로 통일, but 학습 비용 큼
- TALLRec : 범용 LLM을 "좋아할까? (Yes/No)" 지시문으로 튜닝, LoRA로 일부 파라미터만 학습

#### 4-2. 인기 편향과 XAI 문제
- 인기 편향(Popularity Bias) : 소수의 인기 아이템이 추천을 독점하는 현상
    - 데이터 불균형 -> 모델의 편향 강화 -> Rich-get-richer
    - 비인기, 신규 아이템은 노출 기회를 얻기 어렵고 사용자는 비슷한 콘텐츠만 반복해서 추천받음
- 설명가능성(XAI) : 이걸 왜 추천했는지 설명할 수 있는가?
    - 딥러닝 모델은 블랙박스 문제
    - LLM이 만든 설명이 실제 추천 근거와 일치한다는 보장 X
- 비용 문제 : GPU 비용, 실시간 스코어링 서버비, 지속 비용 -> 모든 아이템에 딥러닝 적용은 부담
- 2단계 구조

    <img src="https://developers.google.com/machine-learning/recommendation/images/Process.svg" width="450" style="background-color:white; padding:10px;">

    1. 후보군 생성 - 선형 모델(내용 기반, 협업 필터링, 행렬 분해)로 수백만 개 -> 수백 개
    2. 순위 매기기 - 딥러닝 모델(GNN, Transformer, VAE)로 정교하게 정렬
- 실제 추천 시스템은 단일 모델이 아니라 여러 모델을 단계별로 조합해 비용과 정확도를 함께 확보